In [1]:
# %%
import os
import re
import sys
import csv
import math

from emb_models import *
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.append(parent_dir)
sys.path.append(parent_dir + '/SERGIO')
print(sys.path)

import yaml
import torch
import numpy as np
import torch.nn as nn
import seaborn as sns
from tqdm import tqdm
import networkx as nx
from torch import optim
from typing import List, Any
import pytorch_lightning as pl
import matplotlib.pyplot as plt
import torch.nn.functional as F
from scipy.stats import ttest_ind
from sklearn.manifold import TSNE
from pytorch_lightning import Trainer
from experiment import GRNVAEExperiment
from dataset import GRNVAEDataset
from sklearn.decomposition import PCA
from scipy.sparse.linalg import bicgstab
from torch.utils.data import Dataset, DataLoader
from pytorch_lightning import LightningDataModule
from scipy.spatial.distance import pdist, squareform
from pytorch_lightning.strategies import DDPStrategy
from pytorch_lightning.loggers import TensorBoardLogger
from lightning_lite.utilities.seed import seed_everything
from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint
from sklearn.metrics import roc_auc_score, mean_squared_error, silhouette_score

from GENIE3.GENIE3 import *
nthreads=12

# %% [markdown]
# ### Get DS and Interactions info

# %%
def parse_dataset_name(folder_name):
    pattern1 = r'De-noised_(\d+)G_(\d+)T_(\d+)cPerT_dynamics_(\d+)_DS(\d+)'
    pattern2 = r'De-noised_(\d+)G_(\d+)T_(\d+)cPerT_(\d+)_DS(\d+)'
    match_p1 = re.match(pattern1, folder_name)
    match_p2 = re.match(pattern2, folder_name)
    if match_p1:
        return {
            'number_genes': int(match_p1.group(1)),
            'number_bins': int(match_p1.group(2)),
            'cells_per_type': int(match_p1.group(3)),
            'dynamics': int(match_p1.group(4)),
            'dataset_id': int(match_p1.group(5)),
            'folder_name': folder_name
        }
    if match_p2:
        return {
            'number_genes': int(match_p2.group(1)),
            'number_bins': int(match_p2.group(2)),
            'cells_per_type': int(match_p2.group(3)),
            'dynamics': int(match_p2.group(4)),
            'dataset_id': int(match_p2.group(5)),
            'folder_name': folder_name
        }
    return

def get_datasets():
    datasets = []
    data_sets_dir = '../SERGIO/data_sets'
    for folder_name in os.listdir(data_sets_dir):
        dataset_info = parse_dataset_name(folder_name)
        if dataset_info:
            datasets.append(dataset_info)
    # Include new datasets
    new_datasets = [
        {
            'dataset_id': 1001,
            'dataset_name': 'mESC',
            'expression_file': 'data/raws/mESC-ExpressionData.csv',
            'network_file': 'data/raws/mESC-network.csv',
        },
        {
            'dataset_id': 1002,
            'dataset_name': 'mHSC-E',
            'expression_file': 'data/raws/mHSC-E-ExpressionData.csv',
            'network_file': 'data/raws/mHSC-E-network.csv',
        },
        {
            'dataset_id': 1003,
            'dataset_name': 'mHSC-GM',
            'expression_file': 'data/raws/mHSC-GM-ExpressionData.csv',
            'network_file': 'data/raws/mHSC-GM-network.csv',
        },
        {
            'dataset_id': 1004,
            'dataset_name': 'mHSC-L',
            'expression_file': 'data/raws/mHSC-L-ExpressionData.csv',
            'network_file': 'data/raws/mHSC-L-network.csv',
        },
        {
            'dataset_id': 1005,
            'dataset_name': 'mESC-200',
            'expression_file': 'data/raws/mESC-200-ExpressionData.csv',
            'network_file': 'data/raws/mESC-200-network.csv',
        },
        {
            'dataset_id': 1006,
            'dataset_name': 'mHSC-E-200',
            'expression_file': 'data/raws/mHSC-E-200-ExpressionData.csv',
            'network_file': 'data/raws/mHSC-E-200-network.csv',
        },
    ]
    datasets.extend(new_datasets)
    return datasets

def load_data(dataset_info):
    dataset_id = dataset_info['dataset_id']
    if dataset_id < 1000:
        # Existing datasets
        data_dir = f'../SERGIO/imputation_data_2/DS{dataset_id}/'
        ds_clean_path = os.path.join(data_dir, 'DS6_clean.npy')
        ds_noisy_path = os.path.join(data_dir, 'DS6_45_iter_0.npy')
        if not os.path.exists(ds_clean_path) or not os.path.exists(ds_noisy_path):
            print(f"Data files not found for Dataset {dataset_id}. Skipping.")
            return None, None, None
        ds_clean = np.load(ds_clean_path).astype(np.float32)
        ds_noisy = np.load(ds_noisy_path).astype(np.float32)
        gene_names = None
    else:
        expression_file = dataset_info['expression_file']
        if not os.path.exists(expression_file):
            print(f"Expression file not found for Dataset {dataset_id}. Skipping.")
            return None, None, None
        import pandas as pd
        df = pd.read_csv(expression_file, index_col=0)
        ds_noisy = df.values.astype(np.float32)
        # Not available for new datasets, so set ds_noisy as raw data
        ds_clean = None  
        gene_names = df.index.tolist()
        # print("gene_names: ", gene_names)
    return ds_clean, ds_noisy, gene_names

def load_interactions_info(num_genes, interactions_file):
    gt = np.zeros((num_genes, num_genes))
    with open(interactions_file, 'r') as f:
        lines = f.readlines()
    for line in lines:
        line_list = line.strip().split(',')
        target_index = int(float(line_list[0]))
        num_regs = int(float(line_list[1]))
        for i in range(num_regs):
            try:
                reg_index = int(float(line_list[i + 2]))
                gt[reg_index, target_index] = 1
            except:
                continue
    return gt

# %% [markdown]
# ### Get H and split into G where G = {train, valid} and H/G = {test}

# %%
def load_ground_truth_grn(file_path, num_genes=None, gene_names=None):
    if gene_names is None:
        # Existing datasets: indices
        if num_genes is None:
            with open(file_path, 'r') as f:
                indices = []
                for line in f:
                    source, target = map(int, line.strip().split(','))
                    indices.extend([source, target])
            num_genes = max(indices) + 1
        H = np.zeros((num_genes, num_genes))
        with open(file_path, 'r') as f:
            for line in f:
                source, target = map(int, line.strip().split(','))
                H[source, target] = 1
    else:
        # New datasets: gene names
        num_genes = len(gene_names)
        gene_to_idx = {gene.lower(): idx for idx, gene in enumerate(gene_names)}
        H = np.zeros((num_genes, num_genes))
        with open(file_path, 'r') as f:
            reader = csv.reader(f)
            next(reader, None)
            for line in reader:
                # print(line)
                source_gene, target_gene = line[0], line[1]
                if source_gene in gene_to_idx and target_gene in gene_to_idx:
                    source_idx = gene_to_idx[source_gene]
                    target_idx = gene_to_idx[target_gene]
                    H[source_idx, target_idx] = 1
    return H

['/scratch/ab9738/dfdl_imputation/contrastive_learning', '/scratch/yz5944/miniconda3/envs/bio2/lib/python310.zip', '/scratch/yz5944/miniconda3/envs/bio2/lib/python3.10', '/scratch/yz5944/miniconda3/envs/bio2/lib/python3.10/lib-dynload', '', '/home/yz5944/.local/lib/python3.10/site-packages', '/scratch/ab9738/dfdl_imputation/bni/inferelator', '/scratch/yz5944/miniconda3/envs/infer3/lib/python3.10/site-packages/setuptools/_vendor', '/scratch/yz5944/miniconda3/envs/bio2/lib/python3.10/site-packages/setuptools/_vendor', '/scratch/yz5944/miniconda3/envs/bio2/lib/python3.10/site-packages', '/scratch/yz5944/miniconda3/envs/bio2/lib/python3.10/site-packages/inferelator-0.6.3-py3.10.egg', '/scratch/ab9738/dfdl_imputation', '/scratch/ab9738/dfdl_imputation/SERGIO']


In [10]:
datasets = get_datasets()
for dataset_info in datasets[7:]:
    dataset_id = dataset_info['dataset_id']
    embs = np.load(f'./results/cl/DS{dataset_id}/vae_embeddings.npy')
    print(embs.shape)

(201, 64)
(201, 64)


In [9]:
!pwd

/scratch/ab9738/dfdl_imputation/contrastive_learning
